## 03 - Performance

Trial duzeyi performans metrikleri ve katilimci x kosul birimine toplama.

Girdi: NB02'nin `samples_built` / `episodes` ciktilari, NB01'in `trials_clean`'i.
Cikti: `trial_metrics.parquet`, `participant_condition.parquet`.

Bu notebook **istatistiksel test yapmaz**. Friedman / Wilcoxon ve noise
seviyesi karari NB06'nin isi. Buradaki `dz` ve "kac katilimcida ayni yonde"
sayilari betimleyici etki buyuklugu.

Karara baglanan iki acik soru:
- **2** sIQR_theta / sIQR_omega, RMS'in ustune bilgi getiriyor mu
- **3** Episode suresi mi T/T0 mi, sansurlu episode'lar ne olacak

In [ ]:
%pip install -q pyyaml pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\elifa\Documents\GitHub\NOROM_Inv_Pendulum\.venv\Scripts\python.exe -m pip install --upgrade pip


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ANALYSIS_ROOT = Path.cwd()
if not (ANALYSIS_ROOT / "config.yaml").exists():
    ANALYSIS_ROOT = ANALYSIS_ROOT.parent
sys.path.insert(0, str(ANALYSIS_ROOT))

from src import performance as perf

with open(ANALYSIS_ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

INTERIM_DIR = ANALYSIS_ROOT / config["paths"]["interim_dir"]
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df_samples, episodes, df_trials = perf.load_built(INTERIM_DIR)
print(f"sample  {len(df_samples):,}")
print(f"episode {len(episodes):,}")
print(f"trial   {len(df_trials):,}")
print(f"katilimci {df_samples.participant_id.nunique()}")

sample  846,060
episode 2,017
trial   636
katilimci 12


## 1. Trial duzeyi metrikler

Maske `analysis_include`: active + measurement + qc_pass + focus. Reset
frameleri disarida, her trial tam 1200 sample -- payda sabit.

Dususler **sebebe gore ayriliyor**: `angle` (pole +-60'a vardi) Park'in
Failed'iyla karsilastirilabilir, `track` (cart raydan cikti) Park'ta
karsiligi olmayan ayri bir olay.

In [ ]:
trial_df = perf.trial_metrics(df_samples, episodes, config)

print(f"{len(trial_df)} trial x {trial_df.shape[1]} kolon")
print(f"trial basina sample: {trial_df.n_samples.min()} - {trial_df.n_samples.max()}")
print()
display(trial_df.head(3))

600 trial x 28 kolon
trial basina sample: 1200 - 1200



,participant_id,noise_level_id,noise_sigma,trial_id,trial_order,round_index,n_samples,mae_angle_deg,rms_angle_deg,max_abs_angle_deg,siqr_theta_deg,siqr_omega_deg_s,cart_rms_m,control_effort,falls_per_trial,active_s,stab_time_s,stab_pct,n_episodes,n_episodes_censored,mean_episode_s,mean_T_over_T0,n_episodes_done,mean_episode_s_done,mean_T_over_T0_done,mean_theta0_abs_deg,falls_angle_per_trial,falls_track_per_trial
0,P001,N1,0.02,T004,1,1,1200,12.890435,18.159373,60.7581,8.380312,15.427775,1.213125,0.210988,2.0,20.0004,17.167010,85.835050,3,1,6.666800,2.197955,2.0,7.52515,2.602993,2.146633,2.0,0.0
1,P001,no_noise,0.00,T005,2,1,1200,16.358144,20.912422,61.5771,12.274813,14.948350,1.626366,0.164997,2.0,20.0004,16.783669,83.918345,3,1,6.666767,2.289315,2.0,9.11680,3.106806,3.005367,2.0,0.0
2,P001,N4,0.25,T006,3,1,1200,11.863969,15.625540,61.9434,8.782475,15.303162,0.741103,0.169086,2.0,20.0004,18.983713,94.918565,3,1,6.666833,2.299258,2.0,4.99180,1.731704,2.724800,2.0,0.0


In [ ]:
cols = ["mae_angle_deg", "rms_angle_deg", "siqr_theta_deg", "siqr_omega_deg_s",
        "stab_time_s", "falls_per_trial", "falls_angle_per_trial",
        "falls_track_per_trial", "control_effort", "cart_rms_m",
        "n_episodes", "mean_episode_s", "mean_T_over_T0"]
display(trial_df[cols].describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
mae_angle_deg,600.0,11.979,4.115,3.066,8.818,12.013,14.875,22.468
rms_angle_deg,600.0,15.886,5.547,3.741,11.445,16.257,20.208,28.448
siqr_theta_deg,600.0,8.885,3.285,2.129,6.481,8.695,10.846,22.032
siqr_omega_deg_s,600.0,13.886,6.156,2.348,9.441,12.696,17.012,47.061
stab_time_s,600.0,18.215,1.668,13.117,17.084,18.517,19.900,20.000
falls_per_trial,600.0,1.927,2.425,0.000,0.000,1.000,3.000,14.000
falls_angle_per_trial,600.0,1.708,2.440,0.000,0.000,1.000,2.000,14.000
falls_track_per_trial,600.0,0.218,0.484,0.000,0.000,0.000,0.000,3.000
control_effort,600.0,0.207,0.068,0.052,0.157,0.203,0.255,0.428
cart_rms_m,600.0,1.084,0.525,0.149,0.667,1.077,1.413,2.988


In [ ]:
display(perf.check_fall_consistency(trial_df, df_trials))

,trial,sample_toplami_vs_unity,sebep_toplami_vs_unity,unity_toplam,aci,ray
0,600,600,600,1156,1025.0,131.0


### Kayittaki `within_bounds_time_s` neden kullanilmadi

Unity'nin kolonu failure limitini (60 deg / 5 m) esik aliyor. Trial 20 s ve
neredeyse butun zaman bu limitin icinde geciyor, o yuzden deger her trial'da
tavana yapisik -- koşullari ayirt edemez.

In [ ]:
ref = df_trials[df_trials.practice == 0]
print("Unity within_bounds_time_s:")
print(ref.within_bounds_time_s.describe().round(3).to_string())
print()
thr = config["performance"]["stab_angle_deg"]
print(f"Bizim stab_time_s (|theta| <= {thr:.0f} deg):")
print(trial_df.stab_time_s.describe().round(3).to_string())

Unity within_bounds_time_s:
count    600.000
mean      19.968
std        0.040
min       19.767
25%       19.950
50%       19.983
75%       20.000
max       20.000

Bizim stab_time_s (|theta| <= 30 deg):
count    600.000
mean      18.215
std        1.668
min       13.117
25%       17.084
50%       18.517
75%       19.900
max       20.000


## 2. sIQR gereksiz mi

CLAUDE.md'nin gerekcesi: iki katilimcinin maPA'si ayni olabilir ama biri
cogunlukla +-5 derecede durup ara sira +-50'ye giderken digeri surekli
+-15'te olabilir. RMS birinciyi orantisiz cezalandirir.

Test iki asamali: (a) katilimci ici merkezlenmis korelasyon -- katilimcilar
arasi seviye farki korelasyonu sisirdigi icin within surumu kullaniliyor,
(b) sIQR'i RMS uzerine regres edip artigin kosul profili hala oynuyor mu.
Ikisi birden gecerse metrik RMS'in kopyasi.

In [ ]:
pc = perf.participant_condition(trial_df)
print(f"{len(pc)} hucre = {pc.participant_id.nunique()} katilimci x "
      f"{pc.noise_level_id.nunique()} kosul, hucre basina {pc.n_trials.unique()} trial")

corr_cols = ["mae_angle_deg", "rms_angle_deg", "siqr_theta_deg",
             "siqr_omega_deg_s", "stab_time_s", "falls_per_trial",
             "control_effort", "cart_rms_m"]
print()
print("Katilimci ici merkezlenmis korelasyon:")
display(perf.metric_correlations(pc, corr_cols))

60 hucre = 12 katilimci x 5 kosul, hucre basina [10] trial

Katilimci ici merkezlenmis korelasyon:


,mae_angle_deg,rms_angle_deg,siqr_theta_deg,siqr_omega_deg_s,stab_time_s,falls_per_trial,control_effort,cart_rms_m
mae_angle_deg,1.000,0.982,0.904,0.630,-0.862,0.462,0.463,0.031
rms_angle_deg,0.982,1.000,0.848,0.597,-0.894,0.471,0.485,0.028
siqr_theta_deg,0.904,0.848,1.000,0.579,-0.709,0.350,0.385,0.121
siqr_omega_deg_s,0.630,0.597,0.579,1.000,-0.546,0.352,0.577,0.037
stab_time_s,-0.862,-0.894,-0.709,-0.546,1.000,-0.407,-0.498,-0.069
falls_per_trial,0.462,0.471,0.350,0.352,-0.407,1.000,0.176,-0.336
control_effort,0.463,0.485,0.385,0.577,-0.498,0.176,1.000,-0.015
cart_rms_m,0.031,0.028,0.121,0.037,-0.069,-0.336,-0.015,1.000


In [ ]:
for target in ["siqr_theta_deg", "siqr_omega_deg_s", "mae_angle_deg"]:
    s, prof = perf.redundancy_check(pc, target, "rms_angle_deg", config)
    print(s.to_string())
    display(prof)
    print()

metrik                   siqr_theta_deg
referans                  rms_angle_deg
r_within                          0.848
ham_lineer_kontrast               3.082
artik_lineer_kontrast             0.282
korunan_trend                     0.091
trend_esigi                        0.25
gereksiz                           True


,ham_profil,artik_profil
noise_level_id,,
no_noise,-0.5557,-0.0478
N1,-0.5421,-0.0091
N2,0.1079,-0.1413
N3,0.5516,0.2194
N4,0.4383,-0.0211



metrik                   siqr_omega_deg_s
referans                    rms_angle_deg
r_within                            0.597
ham_lineer_kontrast                 2.695
artik_lineer_kontrast              -0.576
korunan_trend                       0.214
trend_esigi                          0.25
gereksiz                             True


,ham_profil,artik_profil
noise_level_id,,
no_noise,-0.7515,-0.1581
N1,-0.3826,0.2402
N2,0.5957,0.3045
N3,0.2674,-0.1207
N4,0.2710,-0.2658



metrik                   mae_angle_deg
referans                 rms_angle_deg
r_within                         0.982
ham_lineer_kontrast               4.13
artik_lineer_kontrast            0.232
korunan_trend                    0.056
trend_esigi                       0.25
gereksiz                          True


,ham_profil,artik_profil
noise_level_id,,
no_noise,-0.7313,-0.0241
N1,-0.7803,-0.0382
N2,0.3030,-0.0440
N3,0.5297,0.0672
N4,0.6789,0.0391


**Karar: sIQR ikisi de NB06'nin metrik setine girmiyor.**

sIQR_theta RMS'in neredeyse kopyasi (r = 0.85) ve noise trendinin sadece
%9'u artikta kaliyor. sIQR_omega ayri bir konstrukt (r = 0.60 -- CLAUDE.md'nin
"acilikten bagimsiz salinim" beklentisi dogru cikti) ama noise trendinin
%79'unu yine RMS acikliyor ve kalan %21 ters isaretli, yani duzensiz.

Ikisi de `trial_metrics.parquet`'te kaliyor: sIQR_omega NB04'te kontrol
mekanizmasini betimlerken ise yarayabilir. Karar metrigi olarak kullanilmiyor.

Kiyas icin maPA da ayni testten geciriliyor -- o da RMS'in kopyasi (r = 0.98),
yani ikisinden sadece biri raporlanmali.

## 3. Sure olcutu: episode suresi mi T/T0 mi

Ludolph'un T/T0'i trial duzeyinde anlamsiz (trial sabit 20 s, T/T0 = 20/T0,
saf theta0 fonksiyonu). Episode duzeyinde anlamli. Burada iki soru birden:

1. Ham episode suresi mi, T0'a bolunmus hali mi -- hangisi baslangic
   acisindan daha bagimsiz
2. Trial sonunda kesilen (sansurlu) episode'lar dahil mi

Sansur cift tarafli sorun: dahil edilirse en iyi denemeler yapay olarak kisa
gorunur, cikarilirsa iyi katilimcinin en iyi episode'lari tamamen silinir.

In [ ]:
e = episodes[(episodes.practice == 0) & episodes.qc_pass]
print(f"measurement episode: {len(e)}")
print(f"  sansurlu   : {int(e.censored.sum())} ({100*e.censored.mean():.1f}%)")
print(f"  dususle    : {int(e.ended_in_fall.sum())}")
print()
print("Sansurlu vs sansursuz sure:")
display(e.groupby("censored").duration_s.describe().round(2))

measurement episode: 1756
  sansurlu   : 600 (34.2%)
  dususle    : 1156

Sansurlu vs sansursuz sure:


,count,mean,std,min,25%,50%,75%,max
censored,,,,,,,,
False,1156.0,5.01,3.89,0.40,2.18,3.73,6.59,19.82
True,600.0,10.35,8.07,0.02,2.61,7.58,20.00,20.00


In [ ]:
# Once temel soru: episode suresi bagimsiz bir olcut mu?
# Episode'lar trial'i tam kapliyor (toplam 20 s) ve her dusus bir episode
# sinirî. O halde mean_episode_s = 20 / (dusus + 1) olmali.
pred = trial_df.active_s / trial_df.n_episodes
print("mean_episode_s == active_s / n_episodes :",
      f"max sapma {float((trial_df.mean_episode_s - pred).abs().max()):.2e}")
print("n_episodes == falls_per_trial + 1       :",
      bool((trial_df.n_episodes == trial_df.falls_per_trial + 1).all()))
print()
print("corr(mean_episode_s, 20/(falls+1))  =",
      round(trial_df.mean_episode_s.corr(20 / (trial_df.falls_per_trial + 1)), 4))
print("corr(mean_T_over_T0, mean_episode_s) =",
      round(trial_df.mean_T_over_T0.corr(trial_df.mean_episode_s), 4))

mean_episode_s == active_s / n_episodes : max sapma 5.00e-05
n_episodes == falls_per_trial + 1       : True

corr(mean_episode_s, 20/(falls+1))  = 1.0
corr(mean_T_over_T0, mean_episode_s) = 0.9349


`mean_episode_s` dusus sayisinin **birebir yeniden yazilmis hali**:
korelasyon tam 1.0. Sabit 20 s'lik trial'da episode sayisi = dusus + 1
oldugu icin ortalama episode suresi 20/(dusus+1)'den ibaret. Yeni hicbir
bilgi tasimiyor. `mean_T_over_T0` de onunla 0.93 korelasyonlu.

In [ ]:
print("Baslangic acisina duyarlilik (episode duzeyi):")
display(perf.theta0_sensitivity(episodes))
print()
print("Adaylar yan yana (katilimci x kosul duzeyi):")
display(perf.duration_candidate_table(pc, config))

Baslangic acisina duyarlilik (episode duzeyi):


,kume,olcut,n,corr_theta0
0,tum episode,duration_s,1756,-0.075
1,tum episode,duration_over_T0,1756,0.141
2,sansursuz,duration_s,1156,-0.133
3,sansursuz,duration_over_T0,1156,0.125



Adaylar yan yana (katilimci x kosul duzeyi):


,kosul_acilimi,N4_dz,N4_n_kotu,corr_theta0_pc,eksik_hucre
aday,,,,,
mean_episode_s,1.5308,-1.028,11,0.229,0
mean_episode_s_done,1.4728,0.248,6,-0.359,3
mean_T_over_T0,0.7295,-1.074,11,0.645,0
mean_T_over_T0_done,0.6381,0.299,6,-0.198,3


**Karar: bagimsiz bir sure metrigi kullanilmiyor; `falls_per_trial` yeterli
istatistik. T/T0 reddedildi.**

Uc aday da eleniyor, ayri ayri sebeplerle:

- **`mean_episode_s`** dusus sayisinin deterministik donusumu (yukarida),
  ayri bir metrik degil.
- **`mean_T_over_T0`** T0'a bolmek duzeltmiyor, **fazla duzeltiyor**: episode
  duzeyinde ham surenin theta0 korelasyonu -0.075 iken bolunmus hali +0.141,
  katilimci x kosul duzeyinde ise 0.229'a karsi **0.645**. Yani Ludolph'un
  normalizasyonu bizim tasarimimizda kirliligi azaltmiyor, artiriyor.
- **Sansursuz surumler** hayatta kalma yanliligi tasiyor. Sansurlu episode
  demek "trial sonuna kadar dusmedi" demek, yani en iyi denemeler. Onlari
  atinca no_noise'un ortalamasi 12.10 s'den 7.38 s'ye duserek en **dusuk**
  kosul haline geliyor -- sacma bir sonuc. N4 etkisi de bu yuzden isaret
  degistiriyor (dz -1.03 -> +0.25).

Ludolph'un sure temelli olcutunu duzgun kullanmak icin sag sansuru ele alan
bir survival analizi (Kaplan-Meier / Cox) gerekir; 1756 episode'un 600'u
sansurlu, bu goz ardi edilecek bir oran degil. NB03'un kapsami disinda,
gerekirse NB05'te yapilir. Pilot karari icin `falls_per_trial` ayni bilgiyi
tasiyor ve yorumu net.

## 4. Katilimci x kosul

Analiz birimi. Her hucre 10 measurement trial'in ortalamasi.

`baseline_farki` = kosul - no_noise, katilimci basina eslesmis fark.
`dz` = mean(fark) / sd(fark). `n_kotu` = 12 katilimcinin kacinda fark
metrigin kotu yonunde. Hepsi betimleyici, p degeri NB06'da.

In [ ]:
labels = perf.condition_labels(trial_df)
print(" | ".join(labels.values()))
print()
display(perf.condition_table(pc))

no_noise (σ=0.00) | N1 (σ=0.02) | N2 (σ=0.05) | N3 (σ=0.08) | N4 (σ=0.25)



noise_level_id,yon,no_noise,N1,N2,N3,N4
mae_angle_deg,-1,11.247 ±0.978,11.198 ±0.966,12.282 ±0.937,12.508 ±0.895,12.658 ±0.849
rms_angle_deg,-1,14.963 ±1.432,14.917 ±1.326,16.339 ±1.327,16.489 ±1.219,16.721 ±1.201
siqr_theta_deg,-1,8.329 ±0.596,8.343 ±0.663,8.993 ±0.563,9.436 ±0.572,9.323 ±0.539
siqr_omega_deg_s,-1,13.135 ±0.829,13.503 ±1.162,14.482 ±0.846,14.153 ±0.942,14.157 ±1.103
stab_time_s,1,18.366 ±0.398,18.491 ±0.354,18.047 ±0.383,18.082 ±0.361,18.088 ±0.355
falls_per_trial,-1,1.825 ±0.606,1.708 ±0.599,1.975 ±0.67,1.917 ±0.619,2.208 ±0.697
falls_angle_per_trial,-1,1.6 ±0.583,1.467 ±0.579,1.775 ±0.677,1.7 ±0.619,2.0 ±0.691
falls_track_per_trial,-1,0.225 ±0.057,0.242 ±0.073,0.2 ±0.073,0.217 ±0.059,0.208 ±0.051
control_effort,0,0.206 ±0.011,0.209 ±0.012,0.209 ±0.01,0.2 ±0.012,0.208 ±0.013
cart_rms_m,0,1.09 ±0.075,1.126 ±0.069,1.017 ±0.085,1.144 ±0.078,1.043 ±0.075


In [ ]:
for m in ["mae_angle_deg", "stab_time_s", "falls_per_trial",
          "falls_angle_per_trial", "falls_track_per_trial", "control_effort"]:
    print(f"--- {m}  ({perf.METRIC_INFO[m][0]}) ---")
    display(perf.baseline_agreement(pc, m, config))

--- mae_angle_deg  (Mean |theta| (deg)) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,-0.049,-0.030,6,12
N2,1.034,1.176,11,12
N3,1.261,1.172,11,12
N4,1.410,0.959,11,12


--- stab_time_s  (Stabilizasyon suresi (s / 20 s)) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,0.125,0.203,6,12
N2,-0.319,-0.827,9,12
N3,-0.284,-0.606,8,12
N4,-0.278,-0.446,10,12


--- falls_per_trial  (Dusus / trial) ---


...

,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,-0.133,-0.198,6,12
N2,0.175,0.230,9,12
N3,0.100,0.185,6,12
N4,0.400,0.724,9,12


--- falls_track_per_trial  (Ray kaynakli dusus / trial) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,0.017,0.114,5,12
N2,-0.025,-0.115,3,12
N3,-0.008,-0.047,4,12
N4,-0.017,-0.093,6,12


--- control_effort  (Control effort (RMS u)) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,0.003,0.115,NaN,12
N2,0.003,0.132,NaN,12
N3,-0.006,-0.188,NaN,12
N4,0.002,0.064,NaN,12


### Baslangic acisi kirliligi

Randomizasyon sorunu (CLAUDE.md 1): butun katilimcilar ayni sabit RNG
dizisinden okuyor. Kosul ortalamalarinda kalan dengesizlik burada.

In [ ]:
display(pc.groupby("noise_level_id", observed=True)
          .mean_theta0_abs_deg.agg(["mean", "std"]).round(3))

,mean,std
noise_level_id,,
no_noise,3.854,0.611
N1,3.531,0.740
N2,3.730,0.412
N3,3.562,0.523
N4,3.736,0.541


## Cikti

In [ ]:
info = perf.save_outputs(trial_df, pc, INTERIM_DIR)
for k, v in info.items():
    print(f"{k:34} ({v:,} satir)")

trial_metrics.parquet              (600 satir)
participant_condition.parquet      (60 satir)


## Ozet

**Metrik seti (NB06'ya giden).** Yonu net ve birbirinin kopyasi olmayanlar:

| Metrik | Yon | Not |
|---|---|---|
| `mae_angle_deg` | dusuk iyi | RMS ile r = 0.98; ikisinden biri secilmeli, maPA daha okunakli |
| `stab_time_s` | yuksek iyi | kendi esigimiz (30 deg); Unity'nin `within_bounds_time_s`'i tavana yapisik |
| `falls_angle_per_trial` | dusuk iyi | Park'in Failed'iyla karsilastirilabilir olan |
| `control_effort` | belirsiz | tie-breaker, tek basina "iyi/kotu" demiyor |
| `cart_rms_m` | belirsiz | tie-breaker |

**Disarida birakilanlar:** `siqr_theta_deg` ve `siqr_omega_deg_s` (noise
trendini RMS zaten acikliyor -- 2), `mean_episode_s` / `mean_T_over_T0` ve
sansursuz surumleri (dusus sayisinin donusumu ya da theta0 kirli -- 3),
`falls_track_per_trial` (kosulla ilgisiz gorunuyor, dz'ler +-0.12 icinde ve
Park karsilastirmasindan zaten cikariliyor). Hepsi `trial_metrics.parquet`'te
duruyor, sadece karar setinde degil.

**Betimleyici tablo ne diyor.** Tum ana metriklerde ayni sekil: no_noise ile
N1 birbirine yapisik, N2'den itibaren monoton bozulma. maPA'da N2/N3/N4
katilimcilarin 11/12'sinde baseline'dan kotu (dz 0.96-1.18). N1'de fark yok
(dz -0.03, 6/12). Yani **orta seviyede iyilesme, yani U sekli yok** --
stochastic resonance beklentisinin tersi. Testler NB06'da.

**Uyari.** `mean_theta0_abs_deg` kosullar arasi 3.53-3.85 deg araliginda,
en zor baslangiclar no_noise'da. Yanlilik bulgunun aleyhine calisiyor, yani
gozlenen bozulmayi sisirmis olamaz.